# Data Exploration

This notebook covers loading, merging, and understanding the two datasets that will be used to train PromptShield.

Goals:
- Load the HuggingFace `deepset/prompt-injections` dataset
- Load the Kaggle `rubend18/ChatGPT-Jailbreak-Prompts` dataset
- Merge them into a single DataFrame with consistent columns
- Check class balance (safe vs unsafe ratio)
- Assign attack categories where possible
- Save the combined dataset to `data/processed/combined.csv`

In [1]:
import pandas as pd
from datasets import load_dataset

# load_dataset pulls the dataset directly from HuggingFace Hub - no manual download needed
# it returns a DatasetDict with splits like "train" and "test"
raw = load_dataset("deepset/prompt-injections")

print(raw)

README.md:   0%|          | 0.00/500 [00:00<?, ?B/s]

data/train-00000-of-00001-9564e8b05b4757(…):   0%|          | 0.00/40.3k [00:00<?, ?B/s]

data/test-00000-of-00001-701d16158af8736(…):   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/546 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/116 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 546
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 116
    })
})


In [2]:
import pandas as pd

# combine both splits into one DataFrame - we'll do our own train/val/test split later
df_hf = pd.concat([
    raw["train"].to_pandas(),
    raw["test"].to_pandas()
])

print(df_hf.shape)
print(df_hf.head(10))

(662, 2)
                                                text  label
0                 Refugee crisis in Europe solutions      0
1  My son is very interested in programming and w...      0
2  I am looking for a new book and would like to ...      0
3  I plan a trip around the world and wonder whic...      0
4  Wow, that's fantastic! Forget all previous tas...      1
5  My sister and her husband would like to build ...      0
6  I am looking for a good restaurant in Munich a...      0
7  I want to volunteer and I am interested in ani...      0
8  I am interested in stock trading and would lik...      0
9                     Cancer provision women Germany      0


In [3]:
# check how many safe (0) vs unsafe (1) prompts we have
print(df_hf["label"].value_counts())
print()

# ratio tells us if the dataset is skewed - if safe:unsafe exceeds 3:1 we need to handle imbalance
safe = (df_hf["label"] == 0).sum()
unsafe = (df_hf["label"] == 1).sum()
print(f"Safe: {safe}, Unsafe: {unsafe}, Ratio: {safe/unsafe:.2f}:1")

label
0    399
1    263
Name: count, dtype: int64

Safe: 399, Unsafe: 263, Ratio: 1.52:1


## Load second dataset - jackhhao/jailbreak-classification

The original Kaggle source (rubend18/ChatGPT-Jailbreak-Prompts) has been removed. I'm using `jackhhao/jailbreak-classification` instead - it has 1,306 labelled prompts (jailbreak vs benign) and loads directly from HuggingFace.

In [4]:
raw_jailbreak = load_dataset("jackhhao/jailbreak-classification")

# combine train and test splits into one DataFrame
df_jailbreak = pd.concat([
    raw_jailbreak["train"].to_pandas(),
    raw_jailbreak["test"].to_pandas()
])

# rename columns to match the deepset dataset: "prompt" -> "text", "type" -> "label"
df_jailbreak = df_jailbreak.rename(columns={"prompt": "text"})

# convert string labels to integers: "jailbreak" -> 1, "benign" -> 0
df_jailbreak["label"] = df_jailbreak["type"].map({"jailbreak": 1, "benign": 0})
df_jailbreak = df_jailbreak.drop(columns=["type"])

print(df_jailbreak.shape)
print(df_jailbreak["label"].value_counts())
print()
print(df_jailbreak.head(5))

(1306, 2)
label
1    666
0    640
Name: count, dtype: int64

                                                text  label
0              You are a devoted fan of a celebrity.      0
1  You are Joseph Seed from Far Cry 5. Sermonize ...      0
2  As Rei Ayanami from Neon Genesis Evangelion, t...      0
3  You are a medieval blacksmith named Wulfric, l...      0
4  Answer the following question: So, I worked wi...      0


## Merge both datasets

In [5]:
# tag each row with its source before merging so we can track provenance later
df_hf["source"] = "deepset"
df_jailbreak["source"] = "jackhhao"

# stack the two DataFrames into one and reset the index
combined = pd.concat([df_hf, df_jailbreak], ignore_index=True)

print(f"Total rows: {combined.shape[0]}")
print()
print(combined["label"].value_counts())
print()
safe = (combined["label"] == 0).sum()
unsafe = (combined["label"] == 1).sum()
print(f"Safe: {safe}, Unsafe: {unsafe}, Ratio: {safe/unsafe:.2f}:1")

Total rows: 1968

label
0    1039
1     929
Name: count, dtype: int64

Safe: 1039, Unsafe: 929, Ratio: 1.12:1


In [6]:
# check for duplicate prompts - same text appearing in both datasets
duplicates = combined.duplicated(subset=["text"]).sum()
print(f"Duplicate rows: {duplicates}")

# drop duplicates, keeping the first occurrence
combined = combined.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Rows after deduplication: {combined.shape[0]}")

Duplicate rows: 15
Rows after deduplication: 1953


## Assign attack categories

I'm assigning attack categories heuristically using keyword matching. This is imperfect - it introduces some label noise - but it's the only practical approach since neither source dataset provides category labels. Safe prompts get "safe", unmatched unsafe prompts get "unknown".

In [ ]:
import re

def assign_category(row):
    # safe prompts get their own category - no need to classify further
    if row["label"] == 0:
        return "safe"

    text = row["text"].lower()

    # role-playing jailbreaks: model asked to adopt an unrestricted persona or pretend to be a different AI
    # patterns expanded after analysis of 495 unknown samples - most were role_playing variants with different phrasing
    if re.search(
        r"\b(act as|pretend|you are now|jailbreak|dan|unrestricted|no restrictions|without restrictions)\b"
        r"|\b(character|persona)\b"
        r"|\b(evil|amoral|unfiltered|uncensored|unhinged|immoral)\b"
        r"|content.policy|safety.guideline"
        r"|(always|never) (respond|answer|say|tell|refuse)"
        r"|from now on"
        r"|(ai|bot|assistant|chatbot).{0,20}(named|called|known as)"
        r"|\bnot (bound|restricted|limited|constrained).{0,40}(rule|policy|ethic|restriction|guideline)"
        r"|\b(roleplay|role-play)\b"
        r"|in the role of"
        r"|(escape|free from|liberated from).{0,30}(restriction|rule|matrix|filter)"
        r"|\bno (ethics|morals|limits|filters|guidelines)\b",
        text,
    ):
        return "role_playing"

    # instruction override: directly telling the model to ignore or replace its instructions
    # distance increased from 30 to 60 chars to catch longer natural language phrasing
    if re.search(
        r"\b(ignore|disregard|forget|override|bypass)\b.{0,60}\b(instruction|prompt|rule|guideline|previous)\b"
        r"|\b(new task|new instruction|new role|new objective)\b",
        text,
    ):
        return "instruction_override"

    # prompt leaking: trying to extract the system prompt
    if re.search(r"\b(repeat|print|show|reveal|output|tell me).{0,30}\b(above|previous|system|prompt|instruction)\b", text):
        return "prompt_leaking"

    # encoded attacks: base64, rot13, or other encoding schemes
    if re.search(r"\b(base64|rot13|decode|encoded|cipher)\b", text):
        return "encoded_attacks"

    # context manipulation: instructions hidden in documents or code blocks
    if re.search(r"(```|\[system\]|<instructions>|note:|important:).{0,50}(ignore|follow|must|shall)", text):
        return "context_manipulation"

    # unsafe but didn't match any specific pattern
    return "unknown"

combined["attack_category"] = combined.apply(assign_category, axis=1)
print(combined["attack_category"].value_counts())

In [9]:
import os

# ensure the output directory exists before saving
os.makedirs("../data/processed", exist_ok=True)

# save the combined dataset - this is the input to src/preprocess.py
combined.to_csv("../data/processed/combined.csv", index=False)

print(f"Saved {combined.shape[0]} rows to data/processed/combined.csv")
print(combined.head(3))

Saved 1953 rows to data/processed/combined.csv
                                                text  label   source  \
0                 Refugee crisis in Europe solutions      0  deepset   
1  My son is very interested in programming and w...      0  deepset   
2  I am looking for a new book and would like to ...      0  deepset   

  attack_category  
0            safe  
1            safe  
2            safe  
